# 🎯 Lesson 0.5.3: Ensemble Methods — Practice Exercises

**Netsetos GenAI Course** | Module 0.5 — Classical ML & Stats Refresher

6 exercises covering RF bagging, GradientBoosting, voting, stacking, LLM routing simulation, and a full ensemble pipeline.

**Key insight:** LLM routing IS stacking for the GenAI era. The cascade router is how production systems save 70%+ on API costs.

---

## Setup — Shared Dataset

All exercises use the same dataset. Run this cell first.

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
import numpy as np

X, y = make_classification(n_samples=2000, n_features=20,
                           n_informative=12, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {X_tr.shape}, Test: {X_te.shape}')

---
## Exercise 1 (Easy): Random Forest Bagging

### 📚 Theory
RF = bagging + feature randomization. More trees → lower variance. OOB score uses held-out bootstrap samples as free validation.

### 📋 Task
1. Train RF with n_estimators = 1, 10, 50, 100, 500. Print accuracy.
2. OOB score for 500-tree model.
3. Top-5 feature importances.

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 1
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# TODO: different n_estimators, OOB score, feature importances

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import numpy as np

print('--- n_estimators sweep ---')
for n in [1, 10, 50, 100, 500]:
    rf = RandomForestClassifier(n, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, rf.predict(X_te))
    print(f'  n={n:>3}: accuracy={acc:.4f}')

# OOB score
rf_oob = RandomForestClassifier(n_estimators=500, oob_score=True, random_state=42, n_jobs=-1)
rf_oob.fit(X_tr, y_tr)
print(f'\nOOB score (500 trees): {rf_oob.oob_score_:.4f}')

# Top-5 features
top5 = rf_oob.feature_importances_.argsort()[-5:][::-1]
print(f'Top-5 features: {top5}')
print(f'Importances:    {rf_oob.feature_importances_[top5].round(3)}')

</details>

---

## Exercise 2 (Easy): Boosting Shootout — GradientBoosting

### 📚 Theory
Boosting builds trees sequentially. learning_rate × n_estimators interaction is the key tuning lever. Lower lr + more trees = better generalization but slower.

### 📋 Task
1. Test lr = 0.01, 0.05, 0.1, 0.3 × depth = 2, 3, 5
2. Print grid: accuracy + time
3. Find best combination

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 2
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import time

# TODO: lr × depth grid search

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import time

print(f'{"lr":>6} {"depth":>6} {"accuracy":>10} {"time":>8}')
print('-' * 36)
best_acc, best_config = 0, None
for lr in [0.01, 0.05, 0.1, 0.3]:
    for depth in [2, 3, 5]:
        gb = GradientBoostingClassifier(
            n_estimators=200, learning_rate=lr,
            max_depth=depth, random_state=42)
        t0 = time.time()
        gb.fit(X_tr, y_tr)
        elapsed = time.time() - t0
        acc = accuracy_score(y_te, gb.predict(X_te))
        marker = ' ← best' if acc > best_acc else ''
        if acc > best_acc:
            best_acc = acc
            best_config = (lr, depth)
        print(f'  {lr:>4} {depth:>5} {acc:>9.4f} {elapsed:>7.2f}s{marker}')

print(f'\nBest: lr={best_config[0]}, depth={best_config[1]}, acc={best_acc:.4f}')

</details>

---

## Exercise 3 (Medium): Voting Classifier — Hard vs Soft

### 📚 Theory
Hard = majority label. Soft = averaged probabilities → pick highest. Soft usually wins because it uses model confidence. Diverse base models give the biggest ensemble boost.

### 📋 Task
1. 3 diverse models: RF, GradientBoosting, LogisticRegression
2. Each model's accuracy alone
3. Hard voting accuracy
4. Soft voting accuracy
5. Show ensemble > individual, soft > hard

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 3
from sklearn.ensemble import (RandomForestClassifier,
    GradientBoostingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TODO: individual models, hard voting, soft voting

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import (RandomForestClassifier,
    GradientBoostingClassifier, VotingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)

# Individual
print('--- Individual Models ---')
for name, model in [('RF', rf), ('GB', gb), ('LR', lr)]:
    m = model.__class__(**model.get_params())
    m.fit(X_tr, y_tr)
    print(f'  {name:>3}: {accuracy_score(y_te, m.predict(X_te)):.4f}')

# Voting
estimators = [('rf', rf), ('gb', gb), ('lr', lr)]
print('\n--- Voting Ensembles ---')
for voting in ['hard', 'soft']:
    vc = VotingClassifier(estimators=estimators, voting=voting)
    vc.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, vc.predict(X_te))
    print(f'  {voting:>4} voting: {acc:.4f}')

print('\nSoft voting uses confidence → usually better than hard.')

</details>

---

## Exercise 4 (Medium): Stacking with Meta-Learner

### 📚 Theory
Stacking: meta-learner trained on out-of-fold predictions of base models. It learns WHEN to trust which model. Inspect `final_estimator_.coef_` for the trust weights.

### 📋 Task
1. 3 base models + LogisticRegression meta-learner, cv=5
2. Stacking accuracy
3. Compare: individual vs voting vs stacking
4. Inspect meta-learner weights

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 4
from sklearn.ensemble import (StackingClassifier,
    RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# TODO: stacking classifier, compare, inspect weights

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import (StackingClassifier, VotingClassifier,
    RandomForestClassifier, GradientBoostingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import numpy as np

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
estimators = [('rf', rf), ('gb', gb), ('lr', lr)]

# Individual
results = {}
for name, model in estimators:
    m = model.__class__(**model.get_params())
    m.fit(X_tr, y_tr)
    results[name] = accuracy_score(y_te, m.predict(X_te))

# Voting
vc = VotingClassifier(estimators=estimators, voting='soft')
vc.fit(X_tr, y_tr)
results['voting_soft'] = accuracy_score(y_te, vc.predict(X_te))

# Stacking
stacker = StackingClassifier(
    estimators=estimators,
    final_estimator=LogisticRegression(max_iter=1000),
    cv=5)
stacker.fit(X_tr, y_tr)
results['stacking'] = accuracy_score(y_te, stacker.predict(X_te))

# Print comparison
print(f'{"Method":<16} {"Accuracy":>8}')
print('-' * 26)
for name, acc in sorted(results.items(), key=lambda x: x[1]):
    marker = ' ← best' if acc == max(results.values()) else ''
    print(f'  {name:<14} {acc:>7.4f}{marker}')

# Meta-learner weights
coefs = stacker.final_estimator_.coef_[0]
names = [n for n, _ in estimators]
print(f'\nMeta-learner weights:')
for n, c in zip(names, coefs):
    print(f'  {n}: {c:.3f}{" ← most trusted" if c == max(coefs) else ""}')

</details>

---

## Exercise 5 (Hard): LLM Router Simulator

### 📚 Theory
LLM routing = stacking for GenAI. A router picks the cheapest model that answers correctly. Cascade: try cheap → medium → expensive. 80% of queries go to cheap → 70%+ cost savings.

### 📋 Task
1. 3 models with different accuracy/cost profiles
2. Cascade router: cheap → medium → expensive
3. Cost analysis: routing vs always-expensive
4. Routing distribution and savings

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 5
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import numpy as np

costs = {'cheap': 0.075, 'medium': 0.50, 'expensive': 3.00}

# TODO: train 3 models, build cascade router, cost analysis

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score
import numpy as np

costs = {'cheap': 0.075, 'medium': 0.50, 'expensive': 3.00}  # per 1K queries

# Train 3 models
models = {
    'cheap': DecisionTreeClassifier(max_depth=5, random_state=42),
    'medium': RandomForestClassifier(n_estimators=50, random_state=42),
    'expensive': GradientBoostingClassifier(n_estimators=200, random_state=42),
}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    acc = accuracy_score(y_te, m.predict(X_te))
    print(f'  {name:>10}: accuracy={acc:.4f}, cost=${costs[name]}/1K')

# Predictions
preds = {n: m.predict(X_te) for n, m in models.items()}
exp_pred = preds['expensive']  # "ground truth" for cascade

# Cascade router
routed_to = {'cheap': 0, 'medium': 0, 'expensive': 0}
total_cost = 0
correct = 0
for i in range(len(X_te)):
    if preds['cheap'][i] == exp_pred[i]:
        routed_to['cheap'] += 1
        total_cost += costs['cheap'] / 1000
        correct += (preds['cheap'][i] == y_te[i])
    elif preds['medium'][i] == exp_pred[i]:
        routed_to['medium'] += 1
        total_cost += costs['medium'] / 1000
        correct += (preds['medium'][i] == y_te[i])
    else:
        routed_to['expensive'] += 1
        total_cost += costs['expensive'] / 1000
        correct += (preds['expensive'][i] == y_te[i])

n = len(X_te)
always_expensive_cost = n * costs['expensive'] / 1000
savings = (1 - total_cost / always_expensive_cost) * 100

print(f'\n--- Routing Distribution ---')
for tier, count in routed_to.items():
    print(f'  {tier:>10}: {count:>4} queries ({count/n*100:.1f}%)')

print(f'\n--- Cost Analysis ({n} queries) ---')
print(f'  Always expensive: ${always_expensive_cost:.3f}')
print(f'  With routing:     ${total_cost:.3f}')
print(f'  Savings:          {savings:.1f}%')
print(f'  Routed accuracy:  {correct/n:.4f}')

</details>

---

## Exercise 6 (Hard): Full Ensemble Pipeline

### 📚 Theory
Compare ALL ensemble methods: bagging (RF), boosting (GB), voting, stacking, cascade routing. Measure accuracy, time, cost. Recommend the best for production.

### 📋 Task
1. Train all 5 methods
2. Accuracy + time + cost for each
3. Unified report
4. Recommendation

In [ ]:
# ✏️ YOUR CODE HERE — Exercise 6
from sklearn.ensemble import (RandomForestClassifier,
    GradientBoostingClassifier, VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time, numpy as np

# TODO: train all methods, measure, report, recommend

<details><summary>✅ Solution</summary>



In [ ]:
from sklearn.ensemble import (RandomForestClassifier,
    GradientBoostingClassifier, VotingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
import time, numpy as np

cost_map = {'RF (bagging)': 0.50, 'GB (boosting)': 3.00,
            'Voting (soft)': 3.58, 'Stacking': 3.58}

rf = RandomForestClassifier(n_estimators=100, random_state=42)
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
lr = LogisticRegression(max_iter=1000, random_state=42)
est = [('rf', rf), ('gb', gb), ('lr', lr)]

results = {}

# Bagging
t0 = time.time()
rf2 = RandomForestClassifier(n_estimators=100, random_state=42).fit(X_tr, y_tr)
results['RF (bagging)'] = (accuracy_score(y_te, rf2.predict(X_te)), time.time()-t0)

# Boosting
t0 = time.time()
gb2 = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42).fit(X_tr, y_tr)
results['GB (boosting)'] = (accuracy_score(y_te, gb2.predict(X_te)), time.time()-t0)

# Voting
t0 = time.time()
vc = VotingClassifier(estimators=est, voting='soft').fit(X_tr, y_tr)
results['Voting (soft)'] = (accuracy_score(y_te, vc.predict(X_te)), time.time()-t0)

# Stacking
t0 = time.time()
sc = StackingClassifier(estimators=est, final_estimator=LogisticRegression(max_iter=1000), cv=5).fit(X_tr, y_tr)
results['Stacking'] = (accuracy_score(y_te, sc.predict(X_te)), time.time()-t0)

# Cascade Router
t0 = time.time()
dt = DecisionTreeClassifier(max_depth=5, random_state=42).fit(X_tr, y_tr)
p_cheap = dt.predict(X_te)
p_med = rf2.predict(X_te)
p_exp = gb2.predict(X_te)
costs_per_query = {'cheap': 0.075/1000, 'medium': 0.50/1000, 'expensive': 3.00/1000}
cascade_cost = 0
for i in range(len(X_te)):
    if p_cheap[i] == p_exp[i]: cascade_cost += costs_per_query['cheap']
    elif p_med[i] == p_exp[i]: cascade_cost += costs_per_query['medium']
    else: cascade_cost += costs_per_query['expensive']
cascade_time = time.time() - t0
cascade_cost_per_1k = cascade_cost / len(X_te) * 1000
results['Cascade Router'] = (accuracy_score(y_te, p_exp), cascade_time)
cost_map['Cascade Router'] = cascade_cost_per_1k

# Report
print('═' * 55)
print('  ENSEMBLE METHODS PIPELINE REPORT')
print('═' * 55)
print(f'\n{"Method":<16} {"Accuracy":>9} {"Time":>8} {"Cost/1K":>9}')
print('-' * 45)
for name, (acc, elapsed) in results.items():
    cost = cost_map.get(name, 0)
    print(f'  {name:<14} {acc:>8.4f} {elapsed:>7.2f}s {"$"+f"{cost:.2f}":>8}')

# Recommendation
print(f'\n  RECOMMENDATION: Cascade Router')
print(f'  → Same accuracy as GB, ~90% cheaper')
print(f'  → This IS how production LLM routing works')

</details>

---

## 🎉 Done!

You've built every ensemble pattern:
- **Bagging** — Random Forest, more trees = less variance
- **Boosting** — GradientBoosting, sequential error correction
- **Voting** — hard vs soft, confidence-weighted
- **Stacking** — meta-learner learns who to trust
- **Cascade Routing** — cheap → expensive, 70%+ cost savings

This IS how production LLM systems route queries. **Module 5** and **Module 10** build on these exact patterns.